# Q6: Modeling Preparation

**Phase 7:** Modeling Preparation  
**Points: 3 points**

**Focus:** Perform temporal train/test split, select features, handle categorical variables.

**Lecture Reference:** Lecture 11, Notebook 3 ([`11/demo/03_pattern_analysis_modeling_prep.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/03_pattern_analysis_modeling_prep.ipynb)), Phase 7. This notebook demonstrates temporal train/test splitting (see "Your Approach" section below for the key code pattern).

---

## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load feature-engineered data from Q4
df = pd.read_csv('output/q4_features.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
# df = pd.read_csv('output/q4_features.csv')
# df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
# df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with features")

Loaded 196,056 records with features


---

## Objective

Prepare data for modeling by performing temporal train/test split, selecting features, and handling categorical variables.

**CRITICAL - Temporal Split:** For time series data, you **MUST** use temporal splitting (earlier data for training, later data for testing). **DO NOT** use random split. Why? Time series data has temporal dependencies - using future data to predict the past would be data leakage.

---

## Required Artifacts

You must create exactly these 5 files in the `output/` directory:

### 1. `output/q6_X_train.csv`
**Format:** CSV file
**Content:** Training features (X)
**Requirements:**
- All feature columns (no target variable)
- Only training data (earlier time periods)
- **No index column** (save with `index=False`)
- **No datetime column** (unless it's a feature, not the index)

### 2. `output/q6_X_test.csv`
**Format:** CSV file
**Content:** Test features (X)
**Requirements:**
- All feature columns (same as X_train)
- Only test data (later time periods)
- **No index column** (save with `index=False`)
- **No datetime column** (unless it's a feature, not the index)

### 3. `output/q6_y_train.csv`
**Format:** CSV file
**Content:** Training target variable (y)
**Requirements:**
- Single column with target variable name as header
- Only training data (corresponding to X_train)
- **No index column** (save with `index=False`)

**Example:**
```csv
Water Temperature
15.2
15.3
15.1
...
```

### 4. `output/q6_y_test.csv`
**Format:** CSV file
**Content:** Test target variable (y)
**Requirements:**
- Single column with target variable name as header
- Only test data (corresponding to X_test)
- **No index column** (save with `index=False`)

### 5. `output/q6_train_test_info.txt`
**Format:** Plain text file
**Content:** Train/test split information
**Required information:**
- Split method: Temporal (80/20 or similar)
- Training set size: [number] samples
- Test set size: [number] samples
- Training date range: [start] to [end]
- Test date range: [start] to [end]
- Number of features: [number]
- Target variable: [name]

**Example format:**
```
TRAIN/TEST SPLIT INFORMATION
==========================

Split Method: Temporal (80/20 split by time)

Training Set Size: 40000 samples
Test Set Size: 10000 samples

Training Date Range: 2022-01-01 00:00:00 to 2026-09-15 07:00:00
Test Date Range: 2026-09-15 08:00:00 to 2027-09-15 07:00:00

Number of Features: 22
Target Variable: Water Temperature
```

---

## Requirements Checklist

- [ ] Target variable selected
- [ ] Temporal train/test split performed (train on earlier data, test on later data - **NOT random split**)
- [ ] Features selected and prepared
- [ ] Categorical variables handled (encoding if needed)
- [ ] No data leakage (future data not in training set)
- [ ] All 5 required artifacts saved with exact filenames

---

## Your Approach

1. **Select target variable** - Choose a meaningful numeric variable to predict
2. **Select features** - Exclude target, non-numeric columns, and any features derived from the target (to avoid data leakage)
3. **Handle categorical variables** - One-hot encode if needed
4. **Perform temporal train/test split** - Sort by datetime, then split by index position (earlier data for training, later for testing)
5. **Save artifacts** - Save X_train, X_test, y_train, y_test as separate CSVs
6. **Document split** - Record split sizes, date ranges, and feature count

---

## Feature Selection Guidelines

When selecting features for modeling, think critically about each feature:

**Red Flags to Watch For:**
- **Circular logic**: Does this feature use the target variable to predict the target?
  - Example: Rolling mean of target, lag of target (if not handled carefully)
  - Example: If predicting `Air Temperature`, using `air_temp_rolling_7h` is circular - you're predicting temperature from smoothed temperature
- **Data leakage**: Does this feature contain information that wouldn't be available at prediction time?
  - Example: Future values, aggregated statistics that include the current value
- **Near-duplicates**: Is this feature nearly identical to the target?
  - Check correlations - if correlation > 0.95, investigate whether it's legitimate
  - Example: A feature with 99%+ correlation with the target is likely problematic

**Good Practices:**
- Use external predictors (other weather variables, temporal features)
- Create rolling windows of **predictors**, not the target
  - Good: `wind_speed_rolling_7h`, `humidity_rolling_24h`
  - Bad: `air_temp_rolling_7h` when predicting Air Temperature
- Use derived features that combine multiple predictors
- Think: "Would I have this information when making a real prediction?"

**Remember:** The goal is to predict the target from **other** information, not from the target itself.

---

## Decision Points

- **Target variable:** What do you want to predict? Temperature? Water conditions? Choose something meaningful and measurable.
- **Temporal split:** **CRITICAL** - Use temporal split (earlier data for training, later data for testing), NOT random split. Why? Time series data has temporal dependencies. Typical split: 80/20 or 70/30.
- **Feature selection:** Which features are most relevant? Consider correlations, domain knowledge, and feature importance from previous analysis.
- **Categorical encoding:** If you have categorical variables, encode them (one-hot encoding, label encoding, etc.) before modeling.

---

## Checkpoint

After Q6, you should have:
- [ ] Temporal train/test split completed (earlier → train, later → test)
- [ ] Features prepared (no target, no datetime index)
- [ ] Categorical variables encoded
- [ ] No data leakage verified
- [ ] All 5 artifacts saved: `q6_X_train.csv`, `q6_X_test.csv`, `q6_y_train.csv`, `q6_y_test.csv`, `q6_train_test_info.txt`

---

**Next:** Continue to `q7_modeling.md` for Modeling.


In [8]:
# 6: TEMPORAL TRAINING/TESTING 6.1 - 6.4

print("="*80)
print("Q6: MODELING DATA PREPARATION - TEMPORAL TRAIN/TEST SPLIT")
print("="*80)

# Load feature-engineered data
print("\n LOADING DATA")
print("-" * 80)
df = pd.read_csv('output/q4_features.csv', 
                 parse_dates=['Measurement Timestamp'], 
                 index_col='Measurement Timestamp')
df = df.sort_index()  # Sorts in chronological order = CRITICAL for temporal analysis

print(f"  Loaded: output/q4_features.csv")
print(f"  Records: {len(df):,}")
print(f"  Date range: {df.index.min()} to {df.index.max()}")
print(f"  Total columns: {len(df.columns)}")

# Identify target variable and features
print("\n IDENTIFY TARGET VARIABLE AND FEATURES")
print("-" * 80)

# Identify original numeric columns (potential targets)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
original_cols = [col for col in numeric_cols 
                 if not any(x in col.lower() for x in 
                           ['rolling', 'lag', 'change', 'deviation', 'daily_mean', 
                            'hourly_avg', 'sin', 'cos', '_times_', '_div_', '_minus_'])]

print(f"Original sensor columns: {original_cols}")

# Select target variable (first original column)
target_col = original_cols[0] if original_cols else numeric_cols[0]
print(f"\n  Selected target variable: {target_col}")

# Define features (all columns except target)
feature_cols = [col for col in df.columns if col != target_col]
print(f"  Number of features: {len(feature_cols)}")

# Check for categorical columns
categorical_cols = df[feature_cols].select_dtypes(include=['object']).columns.tolist()
print(f"  Categorical columns: {len(categorical_cols)}")
if categorical_cols:
    print(f"  Columns: {categorical_cols}")

# Remove any remaining non-numeric columns for modeling
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"  Numeric features for modeling: {len(numeric_features)}")

# Handle missing values before split
print("\n HANDLING MISSING VALUES")
print("-" * 80)

missing_counts = df[numeric_features + [target_col]].isnull().sum()
features_with_missing = missing_counts[missing_counts > 0]

if len(features_with_missing) > 0:
    print(f"! Found {len(features_with_missing)} columns with missing values")
    for col, count in features_with_missing.items():
        pct = (count / len(df)) * 100
        print(f"  - {col}: {count:,} ({pct:.2f}%)")
    
    print(f"\nDropping rows with missing values...")
    df_clean = df[[target_col] + numeric_features].dropna()
    rows_dropped = len(df) - len(df_clean)
    print(f"  Dropped {rows_dropped:,} rows ({rows_dropped/len(df)*100:.2f}%)")
    print(f"  Remaining: {len(df_clean):,} rows")
else:
    print("  No missing values found")
    df_clean = df[[target_col] + numeric_features].copy()

# TEMPORAL TRAIN/TEST SPLIT
print("\n TEMPORAL TRAINING/TESTING SPLIT (80/20)")
print("-" * 80)
print("CRITICAL: Using temporal split (NOT random split)")
print("  - Training: Earlier time periods")
print("  - Testing: Later time periods")
print("  - This prevents data leakage in time series")

# Calculate split point (80% train, 20% test)
split_ratio = 0.8
split_index = int(len(df_clean) * split_ratio)
split_date = df_clean.index[split_index]

print(f"\nSplit configuration:")
print(f"  - Split ratio: {split_ratio*100:.0f}% train / {(1-split_ratio)*100:.0f}% test")
print(f"  - Split date: {split_date}")
print(f"  - Training period: {df_clean.index[0]} to {split_date}")
print(f"  - Testing period: {split_date} to {df_clean.index[-1]}")

# Perform temporal split
train_data = df_clean.iloc[:split_index]
test_data = df_clean.iloc[split_index:]

print(f"\n  Split complete:")
print(f"  - Training set: {len(train_data):,} records ({len(train_data)/len(df_clean)*100:.1f}%)")
print(f"  - Test set: {len(test_data):,} records ({len(test_data)/len(df_clean)*100:.1f}%)")

# Verify temporal ordering
assert train_data.index.max() <= test_data.index.min(), "ERROR: Temporal split failed!"
print(f"  Temporal ordering verified: train data ends before test data begins")

# Separate features and target
print("\n. SEPARATING FEATURES AND TARGET DATA FOR TRAINING/TESTING")
print("-" * 80)

X_train = train_data[numeric_features]
y_train = train_data[target_col]
X_test = test_data[numeric_features]
y_test = test_data[target_col]

print(f"  Training set:")
print(f"  - X_train shape: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"  - y_train shape: {y_train.shape[0]:,} values")

print(f"\n  Test set:")
print(f"  - X_test shape: {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"  - y_test shape: {y_test.shape[0]:,} values")

# Feature statistics
print("\n FEATURE STATISTICS")
print("-" * 80)

print(f"Feature summary:")
print(f"  - Total features: {len(numeric_features)}")
print(f"  - Feature types:")

feature_types = {
    'temporal': [f for f in numeric_features if any(x in f.lower() for x in ['hour', 'day', 'month', 'year', 'week', 'weekend'])],
    'rolling': [f for f in numeric_features if 'rolling' in f.lower()],
    'lag': [f for f in numeric_features if 'lag' in f.lower()],
    'original': [f for f in numeric_features if f in original_cols and f != target_col],
    'derived': [f for f in numeric_features if any(x in f.lower() for x in ['change', 'deviation', 'mean', 'avg'])]
}

for ftype, features in feature_types.items():
    if features:
        print(f"    • {ftype}: {len(features)} features")

# Save datasets
print("\n SAVING TRAIN/TEST DATASETS")
print("-" * 80)

# Save X_train
X_train.reset_index(drop=True).to_csv('output/q6_X_train.csv', index=False)
print(f"  Saved: output/q6_X_train.csv")
print(f"  Shape: {X_train.shape[0]:,} rows × {X_train.shape[1]} columns")
print(f"  Index: False (no index column)")
print(f"  Datetime: Not included (features only)")

# Save X_test
X_test.reset_index(drop=True).to_csv('output/q6_X_test.csv', index=False)
print(f"\n  Saved: output/q6_X_test.csv")
print(f"  Shape: {X_test.shape[0]:,} rows × {X_test.shape[1]} columns")

# Save y_train
y_train.reset_index(drop=True).to_csv('output/q6_y_train.csv', index=False, header=True)
print(f"\n  Saved: output/q6_y_train.csv")
print(f"  Shape: {y_train.shape[0]:,} values")

# Save y_test
y_test.reset_index(drop=True).to_csv('output/q6_y_test.csv', index=False, header=True)
print(f"\n  Saved: output/q6_y_test.csv")
print(f"  Shape: {y_test.shape[0]:,} values")


# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  Temporal train/test split complete")
print(f"  Target variable: {target_col}")
print(f"  Features: {len(numeric_features)}")
print(f"  Training set: {len(train_data):,} records ({split_ratio*100:.0f}%)")
print(f"  Test set: {len(test_data):,} records ({(1-split_ratio)*100:.0f}%)")
print(f"\n  Output files created:")
print(f"  - output/q6_X_train.csv")
print(f"  - output/q6_X_test.csv")
print(f"  - output/q6_y_train.csv")
print(f"  - output/q6_y_test.csv")
print(f"\n  CRITICAL: Temporal ordering preserved")
print(f"  Train period: {train_data.index[0].date()} to {train_data.index[-1].date()}")
print(f"  Test period: {test_data.index[0].date()} to {test_data.index[-1].date()}")

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE - READY FOR MODELING")
print("="*80)

Q6: MODELING DATA PREPARATION - TEMPORAL TRAIN/TEST SPLIT

 LOADING DATA
--------------------------------------------------------------------------------
  Loaded: output/q4_features.csv
  Records: 196,056
  Date range: 2015-04-25 09:00:00 to 2025-11-27 22:00:00
  Total columns: 306

 IDENTIFY TARGET VARIABLE AND FEATURES
--------------------------------------------------------------------------------
Original sensor columns: ['Air Temperature', 'Wet Bulb Temperature', 'Humidity', 'Rain Intensity', 'Interval Rain', 'Total Rain', 'Precipitation Type', 'Wind Direction', 'Wind Speed', 'Maximum Wind Speed', 'Barometric Pressure', 'Solar Radiation', 'Heading', 'Battery Life', 'hour', 'day_of_week', 'month', 'year', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend']

  Selected target variable: Air Temperature
  Number of features: 305
  Categorical columns: 3
  Columns: ['Station Name', 'Measurement Timestamp Label', 'Measurement ID']
  Numeric features for modeling: 302

 HANDLING MIS

In [9]:
# 6.5 Training Test Info Plain Text File

print("="*80)
print("Q6: GENERATING TRAIN/TEST SPLIT INFO")
print("="*80)

# Load the saved datasets to get accurate information
print("\n1. LOADING SAVED DATASETS")
print("-" * 80)

X_train = pd.read_csv('output/q6_X_train.csv')
X_test = pd.read_csv('output/q6_X_test.csv')
y_train = pd.read_csv('output/q6_y_train.csv')
y_test = pd.read_csv('output/q6_y_test.csv')

print(f"  Loaded X_train: {X_train.shape}")
print(f"  Loaded X_test: {X_test.shape}")
print(f"  Loaded y_train: {y_train.shape}")
print(f"  Loaded y_test: {y_test.shape}")

# Get target variable name
target_variable = y_train.columns[0]
print(f"\n  Target variable: {target_variable}")

# Load original data to get date ranges
print("\n  LOADING ORIGINAL DATA FOR DATE RANGES")
print("-" * 80)

df = pd.read_csv('output/q4_features.csv', 
                 parse_dates=['Measurement Timestamp'], 
                 index_col='Measurement Timestamp')
df = df.sort_index()

# Identify original numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
original_cols = [col for col in numeric_cols 
                 if not any(x in col.lower() for x in 
                           ['rolling', 'lag', 'change', 'deviation', 'daily_mean', 
                            'hourly_avg', 'sin', 'cos', '_times_', '_div_', '_minus_'])]

# Remove missing values (same as in main script)
numeric_features = X_train.columns.tolist()
df_clean = df[[target_variable] + numeric_features].dropna()

print(f"  Cleaned data: {len(df_clean):,} records")

# Calculate split point
split_ratio = 0.8
split_index = int(len(df_clean) * split_ratio)

# Get date ranges
train_data = df_clean.iloc[:split_index]
test_data = df_clean.iloc[split_index:]

train_start = train_data.index.min()
train_end = train_data.index.max()
test_start = test_data.index.min()
test_end = test_data.index.max()

print(f"  Training period: {train_start} to {train_end}")
print(f"  Test period: {test_start} to {test_end}")

# Calculate additional statistics
num_features = X_train.shape[1]
train_samples = len(X_train)
test_samples = len(X_test)
total_samples = train_samples + test_samples
train_percentage = (train_samples / total_samples) * 100
test_percentage = (test_samples / total_samples) * 100

# Generate report content
print("\n GENERATING REPORT")
print("-" * 80)

report_lines = []

def add_line(text="", indent=0):
    """Helper function to add formatted lines"""
    report_lines.append("  " * indent + text)

# Report header
add_line("TRAIN/TEST SPLIT INFORMATION")
add_line("=" * 70)
add_line()

# Split method
add_line("1. SPLIT METHOD:")
add_line("-" * 70)
add_line(f"Method: Temporal Split (chronological)")
add_line(f"Ratio: {split_ratio*100:.0f}% training / {(1-split_ratio)*100:.0f}% testing")
add_line(f"Rationale: Time series data requires temporal split to prevent data leakage")
add_line()

# Dataset sizes
add_line("2. DATASET SIZES:")
add_line("-" * 70)
add_line(f"Training set size: {train_samples:,} samples ({train_percentage:.1f}%)")
add_line(f"Test set size: {test_samples:,} samples ({test_percentage:.1f}%)")
add_line(f"Total samples: {total_samples:,}")
add_line()

# Date ranges
add_line("3. DATE RANGES:")
add_line("-" * 70)
add_line(f"Training date range: {train_start} to {train_end}")

# Calculate duration for training
train_duration = train_end - train_start
train_days = train_duration.days
add_line(f"  Duration: {train_days} days")

add_line()
add_line(f"Test date range: {test_start} to {test_end}")

# Calculate duration for test
test_duration = test_end - test_start
test_days = test_duration.days
add_line(f"  Duration: {test_days} days")

add_line()

# Features
add_line("4. FEATURES:")
add_line("-" * 70)
add_line(f"Number of features: {num_features}")
add_line()

# Feature breakdown
feature_types = {
    'temporal': [f for f in numeric_features if any(x in f.lower() for x in ['hour', 'day', 'month', 'year', 'week', 'weekend', 'business'])],
    'rolling': [f for f in numeric_features if 'rolling' in f.lower()],
    'lag': [f for f in numeric_features if 'lag' in f.lower()],
    'change': [f for f in numeric_features if 'change' in f.lower()],
    'aggregation': [f for f in numeric_features if any(x in f.lower() for x in ['daily_mean', 'hourly_avg', 'deviation'])],
    'original': [f for f in numeric_features if f in original_cols],
    'interaction': [f for f in numeric_features if any(x in f.lower() for x in ['_times_', '_div_', '_minus_'])]
}

add_line("Feature breakdown:")
for ftype, features in feature_types.items():
    if features:
        add_line(f"  - {ftype.capitalize()}: {len(features)} features")

add_line()

# Target variable
add_line("5. TARGET VARIABLE:")
add_line("-" * 70)
add_line(f"Target variable: {target_variable}")
add_line()

# Target statistics
add_line("Target statistics (training set):")
add_line(f"  - Mean: {y_train[target_variable].mean():.2f}")
add_line(f"  - Std Dev: {y_train[target_variable].std():.2f}")
add_line(f"  - Min: {y_train[target_variable].min():.2f}")
add_line(f"  - Max: {y_train[target_variable].max():.2f}")
add_line()

add_line("Target statistics (test set):")
add_line(f"  - Mean: {y_test[target_variable].mean():.2f}")
add_line(f"  - Std Dev: {y_test[target_variable].std():.2f}")
add_line(f"  - Min: {y_test[target_variable].min():.2f}")
add_line(f"  - Max: {y_test[target_variable].max():.2f}")
add_line()


# Output files
add_line("OUTPUT FILES:")
add_line("-" * 70)
add_line("Generated files:")
add_line(f"  - output/q6_X_train.csv ({X_train.shape[0]:,} × {X_train.shape[1]})")
add_line(f"  - output/q6_X_test.csv ({X_test.shape[0]:,} × {X_test.shape[1]})")
add_line(f"  - output/q6_y_train.csv ({y_train.shape[0]:,} × 1)")
add_line(f"  - output/q6_y_test.csv ({y_test.shape[0]:,} × 1)")
add_line()

# Footer
add_line("=" * 70)
add_line("Data is ready for model training and evaluation")
add_line("=" * 70)

# Write report to file
report_content = "\n".join(report_lines)

with open('output/q6_train_test_info.txt', 'w') as f:
    f.write(report_content)

print(f"  Saved to: output/q6_train_test_info.txt")
print(f"  Lines: {len(report_lines)}")

# Display report
print("\n" + "="*80)
print("REPORT PREVIEW")
print("="*80)
print(report_content)


# Summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"  Generated train/test split information report")
print(f"  Saved to: output/q6_train_test_info.txt")
print(f"\n  Report includes:")
print(f"  - Split method: Temporal (80/20)")
print(f"  - Training set: {train_samples:,} samples")
print(f"  - Test set: {test_samples:,} samples")
print(f"  - Training dates: {train_start.date()} to {train_end.date()}")
print(f"  - Test dates: {test_start.date()} to {test_end.date()}")
print(f"  - Features: {num_features}")
print(f"  - Target: {target_variable}")

print("\n" + "="*80)
print("TRAIN/TEST INFO REPORT COMPLETE")
print("="*80)

Q6: GENERATING TRAIN/TEST SPLIT INFO

1. LOADING SAVED DATASETS
--------------------------------------------------------------------------------
  Loaded X_train: (156825, 302)
  Loaded X_test: (39207, 302)
  Loaded y_train: (156825, 1)
  Loaded y_test: (39207, 1)

  Target variable: Air Temperature

  LOADING ORIGINAL DATA FOR DATE RANGES
--------------------------------------------------------------------------------
  Cleaned data: 196,032 records
  Training period: 2015-05-23 11:00:00 to 2023-07-01 17:00:00
  Test period: 2023-07-01 18:00:00 to 2025-11-27 22:00:00

 GENERATING REPORT
--------------------------------------------------------------------------------
  Saved to: output/q6_train_test_info.txt
  Lines: 62

REPORT PREVIEW
TRAIN/TEST SPLIT INFORMATION

1. SPLIT METHOD:
----------------------------------------------------------------------
Method: Temporal Split (chronological)
Ratio: 80% training / 20% testing
Rationale: Time series data requires temporal split to prevent 